# CIFAR-10-LT · PLWCE α-sweep 파일럿

`plwce`의 최적 지수 α\*가 **불균형비(IR)와 어떤 함수 관계**인지 확인하기 위한 분석용 노트북.
메인 실험(`CIFAR10_LT.ipynb`)과 동일한 데이터·모델·학습 설정을 재사용하되, **plwce 한 종만**
α를 촘촘히 sweep 한다. Optuna proxy(20ep·subset)가 아니라 **full-ish epochs**로 돌려 α\* 추정을 안정화.

**설계**
- IR ∈ {10, 20, 50, 100, 200} — 같은 데이터셋에서 IR만 통제변수로 변화
- α ∈ {0.5, 1.0, …, 6.0} (0.5 간격, 12점)
- seeds = {42,43,44} (파일럿 3개), `SWEEP_EPOCHS=100`
- α\* = (seed 평균 test F1-Macro) 곡선의 정점 (top-3 2차 피팅으로 연속 보정)
- α\* vs **IR** 과 vs **log(IR)** 를 각각 1차 피팅 → R² 비교로 함수형 식별

> ⚠️ **비용**: 기본값 5 IR × 12 α × 3 seed × 100ep = **180 full-ish runs**. T4 기준 대략 20~35 GPU-h.
> 먼저 곡선 모양만 보려면 Cell 1의 `QUICK=True` (2 IR × 6 α × 1 seed × 60ep = 12 runs)로 스모크 테스트.
> per-run 체크포인트라 중단/재개 가능. 최종 논문 수치는 `SWEEP_EPOCHS=200`(메인과 동일) 권장.

In [1]:
# === Cell 0: 환경 설정 ===
!pip install torchvision pandas -q

import os, sys, json
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from sklearn.metrics import f1_score

# --- Google Drive 마운트 (Colab only) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Google Drive 마운트 완료')
except Exception:
    IN_COLAB = False
    print('로컬/하이브리드 환경에서 실행 중')

# --- 모듈 경로: custom_losses.py / resnet32.py ---
IMG_CLF_DIR = os.getcwd()
if not os.path.exists(f'{IMG_CLF_DIR}/custom_losses.py'):
    if IN_COLAB:
        IMG_CLF_DIR = '/content/drive/MyDrive/imbalanced-data-LWCE/image_classification'
    else:
        IMG_CLF_DIR = 'C:/Users/Seung/Desktop/Research/Deep_Learning/imbalanced-data-LWCE/image_classification'
if IMG_CLF_DIR not in sys.path:
    sys.path.insert(0, IMG_CLF_DIR)

from custom_losses import get_clf_loss
from resnet32 import build_resnet32
print(f'모듈 로드 성공: {IMG_CLF_DIR}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

# ==================================================================
# 실험 설정 (sweep 파라미터)
# ==================================================================
DATASET     = 'cifar10'
NUM_CLASSES = 10
LOSS_NAME   = 'plwce'                                   # 이 노트북은 plwce 전용

IR_LIST     = [10, 20, 50, 100, 200]                   # IR 통제변수
ALPHA_GRID  = [round(a, 2) for a in np.arange(0.5, 6.0001, 0.5)]   # 0.5..6.0, 12점
SEEDS       = [42, 43, 44]
SWEEP_EPOCHS = 100                                      # proxy(20) < 100 < final(200)

BATCH_SIZE  = 128
NUM_WORKERS = 0
LR          = 0.1
SEED        = 42

# --- 빠른 스모크 테스트 토글 ---
QUICK = False
if QUICK:
    IR_LIST      = [10, 100]
    ALPHA_GRID   = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
    SEEDS        = [42]
    SWEEP_EPOCHS = 60

# --- 결과 경로 ---
if IN_COLAB:
    RESULTS_BASE = '/content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR10_alpha_sweep'
else:
    RESULTS_BASE = './results/CIFAR10_alpha_sweep'
os.makedirs(RESULTS_BASE, exist_ok=True)
CKPT = f'{RESULTS_BASE}/alpha_sweep_checkpoint.json'

np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed(SEED)

n_runs = len(IR_LIST) * len(ALPHA_GRID) * len(SEEDS)
print('\n설정 완료')
print(f'  IR={IR_LIST}')
print(f'  alpha grid({len(ALPHA_GRID)})={ALPHA_GRID}')
print(f'  seeds={SEEDS}, epochs={SWEEP_EPOCHS}')
print(f'  총 {n_runs} runs -> 체크포인트: {CKPT}')

Mounted at /content/drive
Google Drive 마운트 완료
모듈 로드 성공: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification
Device: cuda
  GPU: NVIDIA L4

설정 완료
  IR=[10, 20, 50, 100, 200]
  alpha grid(12)=[np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0), np.float64(5.5), np.float64(6.0)]
  seeds=[42, 43, 44], epochs=100
  총 180 runs -> 체크포인트: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR10_alpha_sweep/alpha_sweep_checkpoint.json


In [2]:
# === Cell 1: CIFAR-LT 데이터 로더 (메인 노트북과 동일 로직) ===

def make_cifar_lt(dataset_name: str, imbalance_ratio: int, seed: int = 42):
    """지수 감소 long-tail: n_i = n_max x IR^(-i/(K-1)). (indices, class_counts) 반환."""
    K = 10 if dataset_name == 'cifar10' else 100
    n_max = 5000 if dataset_name == 'cifar10' else 500
    if dataset_name == 'cifar10':
        dataset = datasets.CIFAR10(root='/tmp/cifar', train=True, download=True, transform=None)
    else:
        dataset = datasets.CIFAR100(root='/tmp/cifar', train=True, download=True, transform=None)
    targets = np.array(dataset.targets)
    class_indices = [np.where(targets == c)[0] for c in range(K)]
    rho = imbalance_ratio ** (-1 / (K - 1))
    class_counts = [int(n_max * (rho ** i)) for i in range(K)]
    np.random.seed(seed)
    lt_indices = []
    for c, n_samples in enumerate(class_counts):
        n_samples = max(1, n_samples)
        selected = np.random.choice(class_indices[c], size=n_samples, replace=False)
        lt_indices.extend(selected)
    lt_indices = np.array(lt_indices)
    np.random.shuffle(lt_indices)
    return lt_indices.tolist(), class_counts


def load_cifar_lt_loaders(ir: int, batch_size: int = 128, num_workers: int = 0):
    """CIFAR-10 LT train(80) / val(20) + 표준 test. 메인 노트북과 동일."""
    full_dataset = datasets.CIFAR10(root='/tmp/cifar', train=True, download=True, transform=None)
    lt_indices, class_counts = make_cifar_lt('cifar10', ir, seed=SEED)
    lt_indices = np.array(lt_indices)
    lt_targets = np.array(full_dataset.targets)[lt_indices]

    train_indices, val_indices = [], []
    for c in range(10):
        c_idx = np.where(lt_targets == c)[0]
        np.random.seed(SEED); np.random.shuffle(c_idx)
        n_c_val = max(1, len(c_idx) // 5)
        val_indices.extend(lt_indices[c_idx[:n_c_val]])
        train_indices.extend(lt_indices[c_idx[n_c_val:]])

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010]),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010]),
    ])
    train_ds = Subset(full_dataset, train_indices); train_ds.dataset.transform = train_tf
    val_ds   = Subset(full_dataset, val_indices)
    test_ds  = datasets.CIFAR10(root='/tmp/cifar', train=False, download=True, transform=test_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader, test_loader, class_counts


# IR별 로더는 한 번만 만들어 캐시 (alpha/seed 루프에서 재사용)
_loader_cache = {}
def get_loaders(ir):
    if ir not in _loader_cache:
        _loader_cache[ir] = load_cifar_lt_loaders(ir, BATCH_SIZE, NUM_WORKERS)
    return _loader_cache[ir]

print('데이터 로더 정의 완료')

데이터 로더 정의 완료


In [3]:
# === Cell 2: 모델 / 평가 / 학습 함수 (메인 노트북과 동일) ===

def compute_test_metrics(model, loader, num_classes, class_counts_train):
    """Test F1-Macro / Balanced / Many.Medium.Few (tertile)."""
    model.eval()
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            for t, p in zip(labels.view(-1), preds.view(-1)):
                cm[t.long(), p.long()] += 1
    per_class_acc = (cm.diagonal().float() / cm.sum(1).clamp(min=1).float()).cpu().numpy()
    y_true, y_pred = [], []
    for i in range(num_classes):
        for j in range(num_classes):
            y_true.extend([i] * cm[i, j].item()); y_pred.extend([j] * cm[i, j].item())
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    counts = np.array(class_counts_train); order = np.argsort(counts)[::-1]; n = len(order)
    return {
        'Top1_Acc': float(cm.diagonal().sum() / cm.sum()),
        'Balanced_Acc': float(per_class_acc.mean()),
        'F1_Macro': float(f1_macro),
        'Many_Acc':   float(per_class_acc[order[:n//3]].mean()),
        'Medium_Acc': float(per_class_acc[order[n//3:2*n//3]].mean()),
        'Few_Acc':    float(per_class_acc[order[2*n//3:]].mean()),
    }


def compute_val_f1(model, loader):
    model.eval(); all_p, all_l = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            all_p.extend(model(imgs.to(device)).argmax(1).cpu().tolist())
            all_l.extend(labels.tolist())
    return f1_score(all_l, all_p, average='macro', zero_division=0)


def train_one(alpha, class_counts, train_loader, val_loader, epochs, seed):
    """plwce 1개 학습 -> val F1 최고 모델 선택해 반환. 스케줄러는 epochs에 맞춰 스케일."""
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    model = build_resnet32(NUM_CLASSES).to(device)
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=2e-4)
    milestones = [int(epochs * 0.8), int(epochs * 0.9)]          # 200ep->[160,180]과 동일 비율
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=0.01)
    criterion = get_clf_loss(LOSS_NAME, class_counts, alpha=alpha, gamma=2.0)

    best_f1, best_state = 0.0, None
    pbar = tqdm(range(epochs), desc=f'a={alpha:.2f} s{seed}', leave=False)
    for _ in pbar:
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward(); optimizer.step()
        vf1 = compute_val_f1(model, val_loader)
        if vf1 > best_f1:
            best_f1 = vf1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        scheduler.step(); pbar.update()
    if best_state:
        model.load_state_dict(best_state)
    return model, best_f1

print('학습/평가 함수 정의 완료')

학습/평가 함수 정의 완료


In [ ]:
# === Cell 3: alpha-sweep 실행 (per-run 체크포인트 재개) ===
os.environ['TQDM_DISABLE'] = '0'

ckpt = json.load(open(CKPT)) if os.path.exists(CKPT) else {}
print(f'체크포인트: {len(ckpt)}/{n_runs} 완료')

for ir in IR_LIST:
    train_loader, val_loader, test_loader, class_counts = get_loaders(ir)
    for alpha in ALPHA_GRID:
        for seed in SEEDS:
            key = f'IR{ir}_a{alpha:.2f}_s{seed}'
            if key in ckpt:
                continue
            model, val_f1 = train_one(alpha, class_counts, train_loader, val_loader,
                                      SWEEP_EPOCHS, seed)
            m = compute_test_metrics(model, test_loader, NUM_CLASSES, class_counts)
            ckpt[key] = {'ir': ir, 'alpha': alpha, 'seed': seed,
                         'val_f1': val_f1, 'metrics': m}
            with open(CKPT, 'w') as f:
                json.dump(ckpt, f, indent=1)
            print(f'  {key}  testF1={m["F1_Macro"]:.4f}  Few={m["Few_Acc"]:.4f}  (valF1={val_f1:.4f})')
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

print('\nsweep 완료')

체크포인트: 32/180 완료


100%|██████████| 170M/170M [00:13<00:00, 12.8MB/s] 


a=5.50 s44:   0%|          | 0/100 [00:00<?, ?it/s]

  IR10_a5.50_s44  testF1=0.8447  Few=0.8300  (valF1=0.8501)


a=6.00 s42:   0%|          | 0/100 [00:00<?, ?it/s]

...

a=5.00 s44:   0%|          | 0/100 [00:00<?, ?it/s]

  IR20_a5.00_s44  testF1=0.8067  Few=0.7697  (valF1=0.8067)


a=5.50 s42:   0%|          | 0/100 [00:00<?, ?it/s]

  IR20_a5.50_s42  testF1=0.8084  Few=0.7770  (valF1=0.8103)


a=5.50 s43:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# === Cell 4: 집계 . alpha*(IR) 추정 . 함수형 피팅 . 시각화 ===
import collections

ckpt = json.load(open(CKPT))
acc = collections.defaultdict(list)
for v in ckpt.values():
    acc[(v['ir'], v['alpha'])].append(v['metrics']['F1_Macro'])

def refine_peak(alphas, means):
    """argmax 인덱스 주변 3점 2차 피팅으로 연속 alpha* 보정 (오목할 때만)."""
    i = int(np.nanargmax(means))
    astar = alphas[i]
    if 0 < i < len(alphas) - 1 and np.all(np.isfinite(means[i-1:i+2])):
        x = np.array(alphas[i-1:i+2]); y = np.array(means[i-1:i+2])
        a2, b2, _ = np.polyfit(x, y, 2)
        if a2 < 0:
            vx = -b2 / (2 * a2)
            if x[0] <= vx <= x[-1]:
                astar = float(vx)
    return astar, i

# 곡선 + alpha*
curves, alpha_star = {}, {}
for ir in IR_LIST:
    means = np.array([np.mean(acc[(ir, a)]) if acc[(ir, a)] else np.nan for a in ALPHA_GRID])
    stds  = np.array([np.std(acc[(ir, a)])  if acc[(ir, a)] else 0.0   for a in ALPHA_GRID])
    curves[ir] = (means, stds)
    alpha_star[ir], _ = refine_peak(ALPHA_GRID, means)

# 함수형 피팅: alpha* vs IR  /  alpha* vs log(IR)
def fit_r2(x, y):
    s, b = np.polyfit(x, y, 1)
    yhat = s * x + b
    ss_res = np.sum((y - yhat) ** 2); ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float('nan')
    return s, b, r2

irs = np.array(IR_LIST, float)
ast = np.array([alpha_star[ir] for ir in IR_LIST])
s_lin, b_lin, r2_lin = fit_r2(irs, ast)
s_log, b_log, r2_log = fit_r2(np.log(irs), ast)

print('IR     alpha*   (mean test-F1 peak)')
for ir in IR_LIST:
    print(f'{ir:5d}  {alpha_star[ir]:.3f}')
print(f'\nalpha* ~ {s_lin:.5f}*IR      + {b_lin:.3f}   |  R2(linear)   = {r2_lin:.3f}')
print(f'alpha* ~ {s_log:.3f}*log(IR) + {b_log:.3f}   |  R2(log-IR)   = {r2_log:.3f}')
better = 'log(IR)' if r2_log >= r2_lin else 'IR(raw)'
print(f'\n-> 더 잘 맞는 축: {better}  (raw-IR 선형이면 plwce의 log 가중과 상충, log-IR 선형이 자연스러움)')

# 결과 저장
out = {'IR_LIST': IR_LIST, 'ALPHA_GRID': ALPHA_GRID, 'SEEDS': SEEDS,
       'SWEEP_EPOCHS': SWEEP_EPOCHS,
       'alpha_star': {int(k): v for k, v in alpha_star.items()},
       'curves': {int(ir): curves[ir][0].tolist() for ir in IR_LIST},
       'fit': {'linear': [s_lin, b_lin, r2_lin], 'log': [s_log, b_log, r2_log]}}
with open(f'{RESULTS_BASE}/alpha_star_summary.json', 'w') as f:
    json.dump(out, f, indent=2)
pd.DataFrame({'IR': IR_LIST, 'alpha_star': ast}).to_csv(
    f'{RESULTS_BASE}/alpha_star.csv', index=False)

# 그림 1: alpha->F1 곡선 (IR별)
fig, ax = plt.subplots(figsize=(8, 5))
cmap = plt.cm.viridis(np.linspace(0, 1, len(IR_LIST)))
for ir, c in zip(IR_LIST, cmap):
    means, stds = curves[ir]
    ax.errorbar(ALPHA_GRID, means, yerr=stds, marker='o', ms=4, capsize=2,
                color=c, label=f'IR={ir}')
    ax.axvline(alpha_star[ir], color=c, ls='--', alpha=0.5)
ax.set_xlabel('plwce alpha'); ax.set_ylabel('Test F1-Macro (mean+-std)')
ax.set_title('CIFAR-10-LT . PLWCE alpha-sweep'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{RESULTS_BASE}/alpha_sweep_curves.png', dpi=130); plt.show()

# 그림 2: alpha* vs IR / log(IR) + 피팅선
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(irs, ast, s=60, zorder=3)
xx = np.linspace(irs.min(), irs.max(), 100)
axes[0].plot(xx, s_lin * xx + b_lin, 'r-', label=f'R2={r2_lin:.3f}')
axes[0].set_xlabel('IR'); axes[0].set_ylabel('alpha*'); axes[0].set_title('alpha* vs IR (raw)')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].scatter(np.log(irs), ast, s=60, zorder=3)
xx2 = np.linspace(np.log(irs).min(), np.log(irs).max(), 100)
axes[1].plot(xx2, s_log * xx2 + b_log, 'r-', label=f'R2={r2_log:.3f}')
axes[1].set_xlabel('log(IR)'); axes[1].set_ylabel('alpha*'); axes[1].set_title('alpha* vs log(IR)')
axes[1].legend(); axes[1].grid(alpha=0.3)
for ir, a in zip(irs, ast):
    axes[0].annotate(f'{int(ir)}', (ir, a), textcoords='offset points', xytext=(5, 5), fontsize=8)
plt.tight_layout(); plt.savefig(f'{RESULTS_BASE}/alpha_star_fit.png', dpi=130); plt.show()
print(f'\n저장: {RESULTS_BASE}/  (alpha_star_summary.json, alpha_star.csv, *.png)')